In [1]:
import duckdb   

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [12]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(15)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10003,Prego,BT10,100,60,z0019_02.csv,2025-10-21 17:47:31.029225
1,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 17:47:31.029225
2,10005,Alicate,BT50,100,300,z0019_02.csv,2025-10-21 17:47:31.029225
3,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:47:31.029225
4,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 17:47:31.029225
5,10008,Lixadeira,BT20,200,125,z0019_02.csv,2025-10-21 17:47:31.029225
6,10003,Prego,BT10,100,60,z0019_02.csv,2025-10-21 17:51:41.437492
7,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 17:51:41.437492
8,10005,Alicate,BT50,100,300,z0019_02.csv,2025-10-21 17:51:41.437492
9,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:51:41.437492


In [11]:
df = con.execute("""
                SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                FROM bronze_z0019
                WHERE data_ingestao >= '2025-10-21'
                """).fetchdf()
df.head(15)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10004,Serrote,BT20,200,200,z0019_01.csv,2025-10-21 17:57:37.539634,1
1,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 17:51:41.437492,2
2,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 17:47:31.029225,3
3,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:51:41.437492,1
4,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:47:31.029225,2
5,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 17:51:41.437492,1
6,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 17:47:31.029225,2
7,10002,Martelo,BT50,100,1500,z0019_01.csv,2025-10-21 17:57:37.539634,1
8,10003,Prego,BT10,100,50,z0019_01.csv,2025-10-21 17:57:37.539634,1
9,10003,Prego,BT10,100,60,z0019_02.csv,2025-10-21 17:51:41.437492,2


In [14]:
df = con.execute("""
                SELECT *
                FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2025-10-21'
                ) WHERE row = 1
                """).fetchdf()
df.head(15)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:51:41.437492,1
1,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 17:51:41.437492,1
2,10005,Alicate,BT50,100,300,z0019_02.csv,2025-10-21 17:51:41.437492,1
3,10003,Prego,BT10,100,50,z0019_01.csv,2025-10-21 17:57:37.539634,1
4,10002,Martelo,BT50,100,1500,z0019_01.csv,2025-10-21 17:57:37.539634,1
5,10004,Serrote,BT20,200,200,z0019_01.csv,2025-10-21 17:57:37.539634,1
6,10001,Parafuso,BT10,100,100,z0019_01.csv,2025-10-21 17:57:37.539634,1
7,10008,Lixadeira,BT20,200,125,z0019_02.csv,2025-10-21 17:51:41.437492,1


In [20]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final =  df_final.rename(columns={"NATBR":"id"})
df_final =  df_final.rename(columns={"MAKTX":"nm_produto"})
df_final =  df_final.rename(columns={"WERKS":"id_categoria"})
df_final =  df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final =  df_final.rename(columns={"LABST":"vl_preco"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10006,Chave de fenda,BT10,100,400
1,10007,Furadeira,BT30,150,75
2,10005,Alicate,BT50,100,300
3,10003,Prego,BT10,100,50
4,10002,Martelo,BT50,100,1500
5,10004,Serrote,BT20,200,200
6,10001,Parafuso,BT10,100,100
7,10008,Lixadeira,BT20,200,125


In [21]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [22]:
df2 = df_final
df2 = df2.astype(
    {
        'id': 'int64',
        'nm_produto': 'string',
        'id_categoria': 'string',
        'id_fornecedor': 'int64',
        'vl_preco': 'float64'
    }
)

In [23]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10006,Chave de fenda,BT10,100,400.0
1,10007,Furadeira,BT30,150,75.0
2,10005,Alicate,BT50,100,300.0
3,10003,Prego,BT10,100,50.0
4,10002,Martelo,BT50,100,1500.0
5,10004,Serrote,BT20,200,200.0
6,10001,Parafuso,BT10,100,100.0
7,10008,Lixadeira,BT20,200,125.0


In [24]:
df2.dtypes

id                        int64
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int64
vl_preco                float64
dtype: object

In [32]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id INT,
        nm_produto STRING,
        id_categoria STRING,
        id_fornecedor INT,
        vl_preco FLOAT
        )
""")

In [33]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10006,Chave de fenda,BT10,100,400.0
1,10007,Furadeira,BT30,150,75.0
2,10005,Alicate,BT50,100,300.0
3,10003,Prego,BT10,100,50.0
4,10002,Martelo,BT50,100,1500.0
5,10004,Serrote,BT20,200,200.0
6,10001,Parafuso,BT10,100,100.0
7,10008,Lixadeira,BT20,200,125.0


In [35]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [36]:
df.resultado = con.execute("SELECT * FROM produtos").fetchdf()
df.resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10006,Chave de fenda,BT10,100,400.0
1,10007,Furadeira,BT30,150,75.0
2,10005,Alicate,BT50,100,300.0
3,10003,Prego,BT10,100,50.0
4,10002,Martelo,BT50,100,1500.0
5,10004,Serrote,BT20,200,200.0
6,10001,Parafuso,BT10,100,100.0
7,10008,Lixadeira,BT20,200,125.0


In [37]:
con.close()